# PA3: dados e qualidade

Este notebook confere a origem, o esquema e a qualidade estrutural do CSV bruto `imdb_top_1000.csv` (catálogo IMDb Top 1000 publicado no Kaggle por Harshit Shankhdhar). Ele **não altera** os dados: apenas lê e mede. As regras de tratamento ficam no notebook 02.

Limite desta etapa: a base traz metadados de filmes e **não** traz interações de usuários (avaliações, históricos). Por isso o recomendador do projeto é por conteúdo.

In [1]:
from pathlib import Path
import sys

def localizar_raiz(inicio=None):
    """Sobe a partir do diretório atual até achar data/raw/imdb_top_1000.csv."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for pasta in [inicio, *inicio.parents]:
        if (pasta / "data" / "raw" / "imdb_top_1000.csv").is_file():
            return pasta
    raise FileNotFoundError("data/raw/imdb_top_1000.csv não encontrado acima de "
                            f"{inicio}. Consulte data/README.md para obter o CSV.")

ROOT = localizar_raiz()
sys.path.insert(0, str(ROOT / "src"))
CSV_BRUTO = ROOT / "data" / "raw" / "imdb_top_1000.csv"
print("Raiz do projeto localizada; CSV bruto:", CSV_BRUTO.relative_to(ROOT))

Raiz do projeto localizada; CSV bruto: data\raw\imdb_top_1000.csv


## Inspeção estrutural

A rotina `inspect_csv` (em `src/inspect_data.py`) exige as colunas indicadas e conta registros, células vazias e duplicatas exatas.

In [2]:
import json
from inspect_data import inspect_csv
## RA: 10356420 - KAYO OLIVEIRA NUKUI ##

COLUNAS_OBRIGATORIAS = ["Series_Title", "Overview", "Genre"]
perfil = inspect_csv(CSV_BRUTO, COLUNAS_OBRIGATORIAS)
print(json.dumps({k: perfil[k] for k in ["file", "rows", "exact_duplicate_excess"]}, ensure_ascii=False))
print("Colunas:", len(perfil["columns"]))
print("Células vazias por coluna (somente colunas com vazio):")
print({c: n for c, n in perfil["empty_cells"].items() if n})

{"file": "imdb_top_1000.csv", "rows": 1000, "exact_duplicate_excess": 0}
Colunas: 16
Células vazias por coluna (somente colunas com vazio):
{'Certificate': 101, 'Meta_score': 157, 'Gross': 169}


## Tipos, nulos e unicidade com pandas

Leitura com os tipos inferidos pelo pandas, apenas para diagnóstico. Os campos `Released_Year`, `Runtime` e `Gross` chegam como texto; o tratamento aparece no notebook 02.

In [3]:
import pandas as pd

bruto = pd.read_csv(CSV_BRUTO)
print("Formato (linhas, colunas):", bruto.shape)
diag = pd.DataFrame({"tipo": bruto.dtypes.astype(str), "nulos": bruto.isna().sum(), "distintos": bruto.nunique()})
diag

Formato (linhas, colunas): (1000, 16)


,tipo,nulos,distintos
Poster_Link,str,0,1000
Series_Title,str,0,999
Released_Year,str,0,100
Certificate,str,101,16
Runtime,str,0,140
Genre,str,0,202
IMDB_Rating,float64,0,17
Overview,str,0,1000
Meta_score,float64,157,63
Director,str,0,548


In [4]:
# Checagens que devem valer para a análise seguinte
assert bruto.shape == (1000, 16)
assert perfil["rows"] == len(bruto) == 1000
assert bruto[["Series_Title", "Overview", "Genre"]].notna().all().all()
assert (bruto["Overview"].str.strip() != "").all()

print("Títulos distintos:", bruto["Series_Title"].nunique(), "de", len(bruto))
print("Linhas totalmente duplicadas:", int(bruto.duplicated().sum()))
print("Pares (título, ano) repetidos:", int(bruto.duplicated(["Series_Title", "Released_Year"]).sum()))
bruto.loc[bruto["Series_Title"].duplicated(keep=False), ["Series_Title", "Released_Year", "Director", "IMDB_Rating"]]

Títulos distintos: 999 de 1000
Linhas totalmente duplicadas: 0
Pares (título, ano) repetidos: 0


,Series_Title,Released_Year,Director,IMDB_Rating
87,Drishyam,2013,Jeethu Joseph,8.3
136,Drishyam,2015,Nishikant Kamat,8.2


## Leitura dos resultados

- A base tem 1000 linhas e 16 colunas; `Series_Title`, `Overview` e `Genre` estão preenchidos em todas as linhas, o que sustenta o uso da sinopse como atributo do recomendador.
- Há 999 títulos distintos: *Drishyam* aparece duas vezes (2013 e 2015, com diretores diferentes). São filmes distintos, não duplicata; por isso o identificador do projeto é a posição da linha (`film_id`), e não o título.
- Não há linhas totalmente duplicadas.
- Os vazios estão em `Certificate`, `Meta_score` e `Gross`. Eles não afetam o recomendador, que usa só a sinopse, e **não** são zeros.

Esta inspeção mede estrutura, não validade do conteúdo (por exemplo, não confere se os gêneros ou as sinopses estão corretos no IMDb).